> **교육 목표** SARIMA로 주간 계절성('7일 전')을 반영해 ARIMA보다 예측이 나아지는지 확인합니다.
>
> **핵심 내용** SARIMA(1,1,1)(1,1,1,7) 학습 → 잔차 확인 → ARIMA와 예측 비교 → 예측 구간

In [1]:
%%html
<style>
table {
    float: left;
    clear: both;
}
</style><!-- Markdown 왼쪽 정렬 -->

# D5_04 SARIMA: 주간 계절성 담기

> **오늘 질문:** '7일 전'을 함께 보면 예측이 얼마나 나아질까?

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비 📋

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
plt.rcParams["axes.unicode_minus"] = False
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

# 1교시에서 한 시간 정리 (00:00 기록 바로잡기 → 구간 시작 시각으로)
se = pd.read_csv("../../data/steel_energy.csv")
se["date"] = pd.to_datetime(se["date"], format="%d/%m/%Y %H:%M")
midnight = (se["date"].dt.hour == 0) & (se["date"].dt.minute == 0)
se.loc[midnight, "date"] += pd.Timedelta(days=1)
se["date"] -= pd.Timedelta(minutes=15)
se = se.set_index("date").sort_index()
daily = se["usage_kwh"].resample("D").sum()

cut = int(len(daily) * 0.8)
train, test = daily.iloc[:cut], daily.iloc[cut:]
arima = ARIMA(train, order=(1, 1, 1)).fit()      # 3교시 모형 (비교용)

## 1. SARIMA(1,1,1)(1,1,1,7) 학습 ✍️

ARIMA 세 숫자에 계절 세 숫자(P, D, Q)와 주기 m = 7을 더합니다. (10초 정도)

> 변수 이름: `sarima`

In [ ]:
# 힌트: SARIMAX(시리즈, order=(p, d, q), seasonal_order=(P, D, Q, m)).fit(disp=False)

## 2. 잔차 확인 ✍️

ARIMA에서 튀던 7, 14일 막대가 사라졌는지 봅니다.

## 3. Test 기간 예측 비교 ✍️

73일을 한 번에 예측해 ARIMA와 나란히 그립니다.

> 변수 이름: `fc_sarima`, `fc_arima`

In [ ]:
# 힌트: 모형.forecast(기간)

## 4. 예측 구간 📋

통계 모형은 예측값과 함께 '이 범위 안에 들 가능성이 95%'인 구간을 줍니다. 처음 3주만 봅니다.

In [ ]:
fc = sarima.get_forecast(21)
ci = fc.conf_int()
plt.figure(figsize=(10, 3.5))
plt.plot(test.index[:21], test.iloc[:21], label="실제", color="black")
plt.plot(ci.index, fc.predicted_mean, label="SARIMA 예측")
plt.fill_between(ci.index, ci.iloc[:, 0], ci.iloc[:, 1], alpha=0.2, label="95% 예측 구간")
plt.legend()
plt.title("SARIMA 예측과 95% 구간 (3주)")
plt.show()

## 정리 💬

**Q1.** SARIMA는 ARIMA보다 무엇이 나아졌나요? AIC와 잔차 ACF, 예측 그래프로 설명해 보세요.

> 답:


**Q2.** 73일 예측의 뒷부분(12월 말)은 어떤가요? 예측 구간은 시간이 갈수록 어떻게 변하나요?

> 답:
